# Parsing de Documentos — Parte 6: Extração Estruturada com LLM

**Problema:** já temos texto limpo (de qualquer etapa anterior).
Agora precisamos transformar texto livre em **dados estruturados** (JSON)
para alimentar bancos de dados, APIs ou o próximo passo do pipeline.

**Ferramenta:** LLM (Gemini) com prompt de extração estruturada.

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import re

In [ ]:
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.0-flash')
print("Gemini configurado.")

---
## Bloco 6a — LLM para extração estruturada

Vamos pegar textos extraídos nas partes anteriores e transformar em JSON.

In [ ]:
# Texto de uma nota fiscal (como seria extraído por OCR + limpeza)
texto_nota = """
NOTA FISCAL Nº 00.123
Emitente: COMÉRCIO SÃO PEDRO LTDA
CNPJ: 12.345.678/0001-99
End: Rua das Flores, 500 - Centro - São Paulo/SP

DESCRIÇÃO                  QTD   UNIT       TOTAL
Cimento CP-II 50kg          10   R$32,00   R$320,00
Areia média (saco 20kg)      5   R$18,00    R$90,00
Tijolo cerâmico (cx 50un)    3   R$95,00   R$285,00
Tinta acrílica branca 18L    2  R$120,00   R$240,00

TOTAL GERAL:                             R$935,00
Forma de pgto: DINHEIRO    Data: 28/03/2026
"""

prompt_json = f"""Extraia os dados da nota fiscal abaixo em formato JSON.

Campos esperados:
- numero_nota (string)
- emitente (string)
- cnpj (string)
- endereco (string)
- data (string, formato ISO 8601: YYYY-MM-DD)
- forma_pagamento (string)
- itens (lista de objetos com: descricao, quantidade (int), valor_unitario (float), valor_total (float))
- total_geral (float)

REGRAS:
- Converta valores monetários para float (R$ 32,00 → 32.00)
- Converta datas para ISO 8601
- Se um campo não existir, use null
- Retorne APENAS o JSON, sem markdown ou explicação

TEXTO DA NOTA FISCAL:
{texto_nota}"""

resposta = model.generate_content(prompt_json)
json_nota = resposta.text

# Limpar possíveis marcadores de código
json_nota = re.sub(r'^```json\s*', '', json_nota.strip())
json_nota = re.sub(r'\s*```$', '', json_nota.strip())

print("=" * 60)
print("EXTRAÇÃO ESTRUTURADA — NOTA FISCAL → JSON")
print("=" * 60)
print(json_nota)

# Validar que é JSON válido
try:
    dados = json.loads(json_nota)
    print("\n✓ JSON válido!")
    print(f"  Emitente: {dados.get('emitente')}")
    print(f"  CNPJ: {dados.get('cnpj')}")
    print(f"  Itens: {len(dados.get('itens', []))}")
    print(f"  Total: {dados.get('total_geral')}")
    print(f"  Data: {dados.get('data')}")
except json.JSONDecodeError as e:
    print(f"\n✗ Erro no JSON: {e}")

In [ ]:
# Texto do relatório de RH (como extraído pelo BS4 na Parte 2)
texto_rh = """
Relatório de Funcionários — Março/2026
Gerado em: 28/03/2026 às 14:30

Equipe de Desenvolvimento:
- Ana Paula Costa, CPF 123.456.789-00, Analista Sênior, salário R$ 8.500,00, status: Ativo
- Bruno Martins, CPF 987.654.321-00, Desenvolvedor Jr, salário R$ 4.200,00, ~~Desligado em 15/02/2026~~
- Carla Rodrigues, CPF 456.789.123-00, Gestora de Produto, salário R$ 12.000,00, Ativo — promovida

Contatos:
- RH: rh@empresa.com.br | Tel: (62) 3333-4444
- TI: suporte@empresa.com.br
- Jurídico: juridico@empresa.com.br
"""

prompt_rh = f"""Extraia os dados dos funcionários do relatório abaixo em JSON.

Estrutura esperada:
{{
  "data_relatorio": "YYYY-MM-DD",
  "funcionarios": [
    {{
      "nome": "string",
      "cpf": "string",
      "cargo": "string",
      "salario": float,
      "status": "ativo" | "desligado" | "promovido",
      "data_desligamento": "YYYY-MM-DD" ou null,
      "observacoes": "string" ou null
    }}
  ],
  "contatos": [
    {{
      "departamento": "string",
      "email": "string",
      "telefone": "string" ou null
    }}
  ]
}}

REGRAS:
- Converta salários para float
- Datas para ISO 8601
- Texto rasurado (~~texto~~) indica desligamento — extraia a data
- Retorne APENAS o JSON

TEXTO:
{texto_rh}"""

resposta_rh = model.generate_content(prompt_rh)
json_rh = resposta_rh.text
json_rh = re.sub(r'^```json\s*', '', json_rh.strip())
json_rh = re.sub(r'\s*```$', '', json_rh.strip())

print("=" * 60)
print("EXTRAÇÃO ESTRUTURADA — RELATÓRIO RH → JSON")
print("=" * 60)
print(json_rh)

try:
    dados_rh = json.loads(json_rh)
    print("\n✓ JSON válido!")
    for f in dados_rh.get('funcionarios', []):
        print(f"  {f.get('nome'):25s} | {f.get('cargo'):20s} | {f.get('status')}")
except json.JSONDecodeError as e:
    print(f"\n✗ Erro: {e}")

A LLM normalizou automaticamente:
- Datas para ISO 8601
- Valores monetários para float
- Campos opcionais como null
- Texto rasurado interpretado como desligamento

---
## Bloco 6b — Comparação: Regex vs LLM na mesma tarefa

O mesmo texto bruto → extrair dados de funcionários.
Regex (manual) vs LLM (automático).

In [ ]:
# === ABORDAGEM 1: REGEX (manual, passo a passo) ===
print("=" * 60)
print("ABORDAGEM 1: REGEX")
print("=" * 60)

# Cada padrão precisa ser escrito manualmente
linhas_func = re.findall(
    r'- (.+?), CPF (\d{3}\.\d{3}\.\d{3}-\d{2}), (.+?), salário (R\$ [\d.,]+), (.+)',
    texto_rh
)

print("\nFuncionários extraídos por regex:")
resultado_regex = []
for nome, cpf, cargo, salario, status_raw in linhas_func:
    # Interpretar status
    if '~~' in status_raw:
        status = 'desligado'
        data_desl = re.search(r'(\d{2}/\d{2}/\d{4})', status_raw)
        data_desl = data_desl.group(1) if data_desl else None
    elif 'promovid' in status_raw.lower():
        status = 'promovido'
        data_desl = None
    else:
        status = 'ativo'
        data_desl = None

    # Converter salário
    salario_float = float(salario.replace('R$ ', '').replace('.', '').replace(',', '.'))

    func = {
        'nome': nome,
        'cpf': cpf,
        'cargo': cargo,
        'salario': salario_float,
        'status': status,
        'data_desligamento': data_desl
    }
    resultado_regex.append(func)
    print(f"  {nome:25s} | {cargo:20s} | {status}")

print(f"\n→ Regex encontrou {len(resultado_regex)} funcionários")
print("→ Foram necessárias ~20 linhas de código com padrões específicos")

In [ ]:
# === ABORDAGEM 2: LLM (já executada acima) ===
print("=" * 60)
print("ABORDAGEM 2: LLM (Gemini)")
print("=" * 60)

# Já temos o resultado em dados_rh
print(f"\nFuncionários extraídos pela LLM:")
for f in dados_rh.get('funcionarios', []):
    print(f"  {f.get('nome', '?'):25s} | {f.get('cargo', '?'):20s} | {f.get('status', '?')}")

print(f"\n→ LLM encontrou {len(dados_rh.get('funcionarios', []))} funcionários")
print("→ Foram necessárias ~5 linhas de código + prompt descritivo")

In [ ]:
# Comparação final
print("=" * 60)
print("REGEX vs LLM — COMPARAÇÃO")
print("=" * 60)
print("""
| Critério               | Regex              | LLM                    |
|------------------------|--------------------|------------------------|
| Linhas de código       | ~20+ (padrões)     | ~5 (prompt)            |
| Tempo de dev           | Minutos a horas    | Segundos               |
| Flexibilidade          | Frágil (formato)   | Robusta (contexto)     |
| Inconsistências        | Quebra             | Interpreta             |
| Custo de execução      | Zero               | $$ (tokens)            |
| Latência               | Microsegundos      | Segundos               |
| Determinismo           | 100%               | Varia entre chamadas   |
| Alucinação             | Impossível         | Possível               |

QUANDO USAR CADA UM:
• Regex: padrões simples e repetitivos em alto volume (CPFs, e-mails, datas)
• LLM: extração semântica de dados variáveis (relatórios, contratos, e-mails)
• Combinação: LLM extrai → regex valida (ex: verificar formato de CPF no JSON)
""")

---
## Trade-offs — Extração Estruturada

| ✅ Vantagens | ❌ Limitações |
|---|---|
| Entende contexto e semântica | Custo por token em escala |
| Normaliza automaticamente | Latência: segundos vs milissegundos |
| Zero configuração de padrões | Pode alucinar (inventar dados) |
| Flexível com qualquer formato | Não-determinístico: respostas variam |

---
## Conclusão — Pipeline Híbrido de Produção

```
Documento chega
  │
  ├─ É HTML?
  │    └─ BeautifulSoup → Regex (limpeza) → texto limpo
  │
  ├─ É PDF nativo?
  │    ├─ Simples (texto corrido) → PyMuPDF → Regex (padrões)
  │    ├─ Com tabelas → pdfplumber → pandas
  │    └─ Complexo (hierarquia) → Docling / Unstructured → Markdown
  │
  ├─ É PDF escaneado / imagem?
  │    └─ Tesseract (OCR) → LLM (correção com imagem) → texto limpo
  │
  └─ Pós-processamento final
       └─ LLM → extração estruturada (JSON) → banco de dados / RAG
```

> **Regex aparece em todas as etapas** como ferramenta de limpeza e extração.
> Nenhuma ferramenta resolve tudo sozinha. O poder está na **combinação**.

| Ferramenta | Melhor para | Evitar quando |
|---|---|---|
| **Regex** | Pós-processamento, padrões em texto limpo | Como ferramenta primária de parsing |
| **BeautifulSoup** | HTML / XML estruturado | Documentos não-HTML |
| **PyMuPDF** | PDF nativo, texto corrido, velocidade | Tabelas ou escaneados |
| **pdfplumber** | Tabelas em PDF nativo | Escaneados ou docs sem bordas |
| **Docling** | Docs complexos com hierarquia (RAG) | Scripts simples e rápidos |
| **Unstructured** | Particionamento automático de docs variados | Quando precisa de controle fino |
| **Tesseract** | OCR gratuito / offline | Precisão crítica sem pós-processamento |
| **LLM (visão)** | OCR de tabelas, docs difíceis | Volume alto com orçamento limitado |
| **Tesseract + LLM** | OCR em produção com qualidade | Orçamento zero |
| **LLM (texto)** | Extração estruturada, correção, normalização | Volume massivo sem budget |